# Phần 1. Hiểu bài toán kinh doanh: dự đoán phí vận chuyển Olist

## 1.1. Ngữ cảnh kinh doanh và lý do chọn đề tài

**Ngữ cảnh kinh doanh**

- Trong thương mại điện tử, phí vận chuyển là một thành phần của tổng chi phí mua hàng và có thể ảnh hưởng đến quyết định hoàn tất giao dịch.
- Theo khảo sát của Baymard Institute vào tháng 2/2024 trên 1.012 người trưởng thành tại Mỹ, 48% người được khảo sát từ bỏ giỏ hàng ở bước thanh toán do chi phí phát sinh (vận chuyển, thuế, phụ phí) quá cao [1].
- Tại Brazil, báo cáo *NuvemCommerce 2026* của Nuvemshop, dẫn số liệu từ khảo sát E-Consumidor, ghi nhận 57% người tiêu dùng được khảo sát cho biết phí vận chuyển quá cao là lý do từ bỏ mua hàng trực tuyến [2].
- Olist là nền tảng thương mại điện tử tại Brazil, kết nối người bán với khách hàng thông qua các đối tác vận chuyển [3].
- Dữ liệu công khai của Olist có thông tin về sản phẩm, vị trí người bán, địa điểm nhận hàng và phí vận chuyển ở cấp sản phẩm, nhưng không có bảng giá chi tiết hoặc thông tin cụ thể về hãng vận chuyển [3].

**Vấn đề cốt lõi cần giải quyết**

- Do dữ liệu Olist không có bảng giá của hãng vận chuyển, chưa rõ yếu tố nào của sản phẩm và tuyến giao hàng liên quan đến phí vận chuyển, và liên quan mạnh đến đâu.

**Giá trị và ý nghĩa**

- Mô hình không thay thế biểu phí thực tế mà cung cấp mức phí tham chiếu.
- Người bán dùng mức phí này để ước tính chi phí vận chuyển của sản phẩm trước khi đăng bán và tổng chi phí khách hàng phải thanh toán.

**Lý do chọn đề tài**

- Bộ dữ liệu Olist gồm khoảng 100.000 đơn hàng trong giai đoạn 2016–2018 [3], có biến `freight_value` cùng thông tin về sản phẩm và địa lý.
- Đây là cơ sở phù hợp để phân tích các yếu tố liên quan và đánh giá khả năng dự báo phí vận chuyển từ dữ liệu lịch sử.

## 1.2. Mục tiêu dự án

**Mục tiêu chung**

Xác định và đo mức liên hệ giữa các đặc điểm của sản phẩm, khoảng cách địa lý, đặc điểm đơn hàng và phí vận chuyển; đồng thời đánh giá mức độ có thể dự đoán phí của một sản phẩm trong một đơn hàng bằng mô hình hồi quy từ các thông tin có sẵn trước khi đặt hàng.

**Mục tiêu cụ thể**

1. Thu thập và làm sạch dữ liệu giao dịch của sàn Olist. Bảng dùng cho mô hình gồm các bảng đơn hàng, sản phẩm, người bán, khách hàng và định vị địa lý; bảng thanh toán và bảng đánh giá không dùng vì chứa thông tin có sau thời điểm phí được định.
2. Phân tích mức liên hệ giữa phí vận chuyển với đặc điểm vật lý của sản phẩm, khoảng cách địa lý và đặc điểm đơn hàng (số sản phẩm trong đơn), bằng thống kê mô tả và mô hình; kết quả là mối liên hệ quan sát được, không phải quan hệ nhân quả.
3. Xây dựng, huấn luyện và đánh giá mô hình hồi quy dự đoán phí vận chuyển trên tập kiểm tra chia theo thời gian, so với mô hình mốc.
4. Xác định phần phí mà các yếu tố trên không giải thích được, để chỉ ra giới hạn của dữ liệu hiện có.
5. Trực quan hóa kết quả bằng biểu đồ để trình bày mối liên hệ giữa đặc điểm sản phẩm, tuyến giao hàng và phí vận chuyển.

## 1.3. Các câu hỏi kinh doanh cốt lõi

**Câu hỏi 1:** Trong các yếu tố gồm cân nặng, thể tích, khoảng cách địa lý, việc người bán và khách hàng ở cùng bang, và danh mục sản phẩm, yếu tố nào có mối liên hệ mạnh nhất với phí vận chuyển của một sản phẩm?

**Câu hỏi 2:** Người bán có thể dự báo phí vận chuyển của một sản phẩm đến một điểm đến xác định, trước khi đặt hàng, với sai số trung bình bao nhiêu BRL? Độ chính xác này có đủ để dùng làm mức phí tham chiếu hay không?

**Câu hỏi 3:** Với cùng sản phẩm, cùng người bán và cùng bang của khách hàng, phí vận chuyển ghi nhận cho sản phẩm trong đơn nhiều sản phẩm khác thế nào so với đơn chỉ có một sản phẩm? (Nguồn dữ liệu ghi phí của đơn nhiều sản phẩm được chia cho các sản phẩm.)

## 1.4. Định nghĩa bài toán Khoa học Dữ liệu

**Loại bài toán**

Bài toán học có giám sát với nhiệm vụ hồi quy dự báo `freight_value`, kết hợp phân tích mô tả để tìm hiểu mối liên hệ giữa đặc điểm sản phẩm, tuyến giao hàng và phí vận chuyển.

**Cơ sở lựa chọn phương pháp**

- `freight_value` là biến số liên tục, tính bằng real Brazil (BRL), nên hồi quy phù hợp với mục tiêu dự báo giá trị phí vận chuyển.
- Phân tích mô tả hỗ trợ trả lời các câu hỏi về mối liên hệ giữa các yếu tố và mức phí.

**Đơn vị quan sát**

Một sản phẩm trong một đơn hàng, xác định bởi cặp khóa `(order_id, order_item_id)`.

**Biến mục tiêu (y)**

`freight_value`: phí vận chuyển ghi nhận cho một sản phẩm, tính bằng BRL.

**Biến đầu vào (X)**

- **Đặc điểm sản phẩm:** cân nặng, ba kích thước, thể tích suy ra, danh mục và giá sản phẩm.
- **Đặc điểm tuyến giao hàng:** bang của người bán và bang của điểm đến (trong dữ liệu là nơi khách hàng nhận hàng), tọa độ địa lý, khoảng cách đường chim bay và biến xác định hai bên có cùng bang hay không.

**Phạm vi và cách chia dữ liệu**

- **Phạm vi:** chỉ xét các đơn hàng đã giao thành công; kết quả vì vậy không mặc nhiên đại diện cho toàn bộ đơn hàng trên nền tảng.
- **Cách chia:** theo thời gian đặt hàng, 80% đơn hàng sớm nhất dùng để huấn luyện và 20% đơn hàng muộn nhất dùng để kiểm tra.
- **Vai trò của thời điểm đặt hàng:** chỉ phục vụ việc chia tập, không làm biến đầu vào.
- **Đơn nhiều sản phẩm:** các sản phẩm thuộc cùng một đơn hàng phải nằm trong cùng một tập dữ liệu.

**Chỉ số đánh giá**

- **MAE (BRL):** sai số tuyệt đối trung bình, phản ánh mức sai lệch dự báo trung bình.
- **RMSE (BRL):** căn bậc hai của sai số bình phương trung bình, nhấn mạnh hơn các dự báo có sai số lớn.
- **R²:** phản ánh mức độ mô hình giải thích biến thiên của biến mục tiêu; giá trị âm cho thấy mô hình kém hơn mốc dự báo bằng trung bình theo tiêu chí này.
- **Mốc đối chứng:** dự báo bằng trung vị `freight_value` của tập huấn luyện. Mô hình được xem là cải thiện khi MAE và RMSE thấp hơn trên cùng tập kiểm tra.

## Tài liệu tham khảo

[1] Baymard Institute, khảo sát ngày 7/2/2024 (1.012 người Mỹ từ 18 tuổi trở lên), trích trong: eMarketer, "Extra costs are the No. 1 reason consumers abandon online carts". https://www.emarketer.com/content/extra-costs-are-the-top-reason-consumers-abandon-online-carts (truy cập ngày 27/10/2026)

[2] Nuvemshop, *NuvemCommerce 2026*, mục Indicadores-chave do e-commerce (số liệu dẫn từ khảo sát E-Consumidor). https://site.nuvemshop.com.br/hubfs/%5BBR%5D%20Descarreg%C3%A1veis/NuvemCommerce%202026.pdf (truy cập ngày 27/10/2026)

[3] Olist, *Brazilian E-Commerce Public Dataset by Olist*, Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data (truy cập ngày 28/10/2026)